Airlines Data Engineering Pipeline

1. Project Overview

This project aims to build a data engineering pipeline for airline data that involves ingestion, profiling, cleaning, transformation, validation, and integration of flight, booking, payment, and passenger data sets. This pipeline will solve problems associated with data quality such as data missingness, duplicates, formatting inconsistencies, and flight duration inconsistencies by taking care of overnight flights in a correct manner. Personal information would be masked while transforming data. The final processed data would be in the form that could be utilized for reporting and creating a Power BI dashboard comprising of relevant airline information.

2. Import Libraries

In [4]:
import pandas as pd  # reading Excel, DataFrames, cleaning and transformation
import numpy as np  # numerical operations

from pathlib import Path  # managing file/folder paths
from datetime import datetime, timedelta  # flight date/time and duration calculations
import hashlib  # hashing PII later
import logging  # recording pipeline events/errors

3. Load Raw Data

In [5]:
file_path = "/content/UseCase - Airlines.xlsx"

In [6]:
flights_df = pd.read_excel(file_path, sheet_name="flights")
payments_df = pd.read_excel(file_path, sheet_name="payments")
bookings_df = pd.read_excel(file_path, sheet_name="bookings")
passengers_df = pd.read_excel(file_path, sheet_name="passengers")

In [7]:
print("First 5 Rows of The Flight Details!")
display(flights_df.head())

print("\n\nFirst 5 Rows of The Payment Details!")
display(payments_df.head())

print("\n\nFirst 5 Rows of The Booking Details!")
display(bookings_df.head())

print("\n\nFirst 5 Rows of The Passenger Details!")
display(passengers_df.head())

First 5 Rows of The Flight Details!


,flight_id,airline,source,destination,departure_time,arrival_time,duration
0,SJ010,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,02:54:00
1,AI155,Air India,BOM,CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,01:48:00
2,UK094,Vistara,BOM,CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,01:45:00
3,AI245,Air India,BOM,CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,02:36:00
4,AI192,Air India,MAA,BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,04:59:00




First 5 Rows of The Payment Details!


,payment_id,booking_id,amount,payment_method
0,PAY1000,B1116,9883.49,NETBANKING
1,PAY1001,B1738,8457.96,NETBANKING
2,PAY1002,B1873,6495.37,UPI
3,PAY1003,B1914,5079.38,NETBANKING
4,PAY1004,B1967,12518.31,CARD




First 5 Rows of The Booking Details!


,booking_id,passenger_id,flight_id,booking_date,status,passport_number,seat_number,emergency_contact_name,emergency_contact_phone
0,B1000,P1591,AI192,2025-06-14 11:37:36.951,CANCELLED,P1945887,3D,Isaac Bakshi,+91-6478475128
1,B1001,P1803,6F026,2025-11-02 11:37:36.951,CANCELLED,L3482012,18A,Anvi Konda,+91-6647078662
2,B1002,P1083,SJ010,2025-08-25 11:37:36.951,CANCELLED,G8507659,30C,Udant Dewan,+91-8405938220
3,B1003,P1364,AI069,2025-12-30 11:37:36.951,CONFIRMED,M0891776,33A,Harsh Chahal,+91-6264636839
4,B1004,P1885,UK003,2025-10-02 11:37:36.951,PENDING,N5742231,25C,Pahal Balay,+91-9336478266




First 5 Rows of The Passenger Details!


,passenger_id,first_name,last_name,age,gender,email,phone,aadhaar_id,date_of_birth
0,P1000,Vivaan,Chatterjee,52,F,vivaan.chatterjee@gmail.com,+91-6896233790,433218196001,1974-04-08
1,P1001,Krishna,Reddy,15,M,krishna.reddy@hotmail.com,+91-6702632297,386379402654,2011-03-07
2,P1002,Myra,Naidu,72,M,myra.naidu@outlook.com,+91-6199585092,615594078161,1954-09-10
3,P1003,Myra,Mishra,61,F,myra.mishra@hotmail.com,+91-8719927151,310341316475,1965-03-12
4,P1004,Saanvi,Banerjee,21,M,saanvi.banerjee@outlook.com,+91-7819595113,419283276483,2005-11-11


4. Data Profiling

In [8]:
# 4.1: Basic Structure

dataframes = {
    "Flights": flights_df,
    "Payments": payments_df,
    "Bookings": bookings_df,
    "Passengers": passengers_df
}

for name, df in dataframes.items():
    print(f"\n{name} Details!")
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    print("Column Names:", df.columns.tolist())


Flights Details!
Rows: 1020
Columns: 7
Column Names: ['flight_id', 'airline', 'source', 'destination', 'departure_time', 'arrival_time', 'duration']

Payments Details!
Rows: 1000
Columns: 4
Column Names: ['payment_id', 'booking_id', 'amount', 'payment_method']

Bookings Details!
Rows: 1000
Columns: 9
Column Names: ['booking_id', 'passenger_id', 'flight_id', 'booking_date', 'status', 'passport_number', 'seat_number', 'emergency_contact_name', 'emergency_contact_phone']

Passengers Details!
Rows: 1039
Columns: 9
Column Names: ['passenger_id', 'first_name', 'last_name', 'age', 'gender', 'email', 'phone', 'aadhaar_id', 'date_of_birth']


In [9]:
# 4.2: Data Types

for name, df in dataframes.items():
    print(f"\nDatatype of {name} Details!")
    print(df.dtypes)


Datatype of Flights Details!
flight_id                 object
airline                   object
source                    object
destination               object
departure_time    datetime64[ns]
arrival_time      datetime64[ns]
duration                  object
dtype: object

Datatype of Payments Details!
payment_id        object
booking_id        object
amount            object
payment_method    object
dtype: object

Datatype of Bookings Details!
booking_id                         object
passenger_id                       object
flight_id                          object
booking_date               datetime64[ns]
status                             object
passport_number                    object
seat_number                        object
emergency_contact_name             object
emergency_contact_phone            object
dtype: object

Datatype of Passengers Details!
passenger_id             object
first_name               object
last_name                object
age                       in

In [10]:
# 4.3: Missing Values

for name, df in dataframes.items():
    print(f"\nMissing values of {name}!")

    missing = df.isnull().sum()
    missing_percentage = (missing / len(df)) * 100

    profiling = pd.DataFrame({
        "Missing Count": missing,
        "Missing Percentage": missing_percentage.round(2)
    })

    display(profiling)


Missing values of Flights!


,Missing Count,Missing Percentage
flight_id,0,0.00
airline,41,4.02
source,0,0.00
destination,0,0.00
departure_time,0,0.00
arrival_time,0,0.00
duration,0,0.00



Missing values of Payments!


,Missing Count,Missing Percentage
payment_id,0,0.0
booking_id,0,0.0
amount,48,4.8
payment_method,0,0.0



Missing values of Bookings!


,Missing Count,Missing Percentage
booking_id,0,0.0
passenger_id,0,0.0
flight_id,0,0.0
booking_date,0,0.0
status,45,4.5
passport_number,0,0.0
seat_number,0,0.0
emergency_contact_name,0,0.0
emergency_contact_phone,0,0.0



Missing values of Passengers!


,Missing Count,Missing Percentage
passenger_id,0,0.00
first_name,0,0.00
last_name,10,0.96
age,0,0.00
gender,0,0.00
email,0,0.00
phone,0,0.00
aadhaar_id,0,0.00
date_of_birth,0,0.00


In [11]:
# 4.4: Duplicate Records

for name, df in dataframes.items():
    duplicate_count = df.duplicated().sum()

    print(f"{name}: {duplicate_count} duplicate rows")

Flights: 15 duplicate rows
Payments: 0 duplicate rows
Bookings: 0 duplicate rows
Passengers: 0 duplicate rows


In [12]:
# Checking duplicate IDs

print("Flight IDs:", flights_df["flight_id"].duplicated().sum())
print("Payment IDs:", payments_df["payment_id"].duplicated().sum())
print("Booking IDs:", bookings_df["booking_id"].duplicated().sum())
print("Passenger IDs:", passengers_df["passenger_id"].duplicated().sum())

Flight IDs: 16
Payment IDs: 0
Booking IDs: 0
Passenger IDs: 39


In [13]:
# 4.5: Numerical Profiling

print("Flights")
display(flights_df.describe())

print("\nPayments")
display(payments_df.describe())

print("\nBookings")
display(bookings_df.describe())

print("\nPassengers")
display(passengers_df.describe())

Flights


,departure_time,arrival_time
count,1020,1020
mean,2026-04-19 06:45:29.350427136,2026-04-19 09:28:24.534786048
min,2026-04-17 12:25:41.701000,2026-04-17 14:10:41.703000
25%,2026-04-18 10:43:26.701250048,2026-04-18 13:13:56.701250048
50%,2026-04-19 06:26:11.701499904,2026-04-19 09:19:11.852000
75%,2026-04-20 03:20:26.703250176,2026-04-20 06:05:41.702749952
max,2026-04-20 23:38:41.701000,2026-04-21 04:04:41.703000



Payments


,payment_id,booking_id,amount,payment_method
count,1000,1000,952,1000
unique,1000,637,922,3
top,PAY1999,B1663,INVALID,UPI
freq,1,6,30,358



Bookings


,booking_date
count,1000
mean,2025-10-15 17:40:29.751507456
min,2025-04-17 11:37:36.951000
25%,2025-07-16 11:37:36.951000064
50%,2025-10-15 11:37:36.952000
75%,2026-01-15 11:37:36.951000064
max,2026-04-17 11:37:36.951000



Passengers


,age,aadhaar_id,date_of_birth
count,1039.000000,1.039000e+03,1039
mean,43.304139,4.912431e+11,1983-03-15 10:41:41.636188672
min,1.000000,2.255794e+09,1937-01-11 00:00:00
25%,20.000000,2.388298e+11,1962-08-02 12:00:00
50%,43.000000,4.817767e+11,1983-05-06 00:00:00
75%,64.000000,7.363137e+11,2006-02-21 00:00:00
max,89.000000,9.992192e+11,2025-12-15 00:00:00
std,25.900616,2.923099e+11,NaN


In [14]:
# 4.6: Categorical Values

categorical_columns = {
    "Flights": ["airline", "source", "destination"],
    "Payments": ["payment_method"],
    "Bookings": ["status"],
    "Passengers": ["gender"]
}

for dataset_name, columns in categorical_columns.items():

    print(dataset_name)
    df = dataframes[dataset_name]
    for column in columns:
        print(f"\n{column}:")
        print(df[column].value_counts(dropna=False))

Flights

airline:
airline
IndiGo       249
SpiceJet     240
Air India    236
Vistara      223
NaN           41
UNKNOWN       31
Name: count, dtype: int64

source:
source
BOM    207
HYD    180
CCU    174
DEL    163
MAA    157
BLR    139
Name: count, dtype: int64

destination:
destination
DEL    200
CCU    188
BOM    174
BLR    165
MAA    151
HYD    142
Name: count, dtype: int64
Payments

payment_method:
payment_method
UPI           358
CARD          329
NETBANKING    313
Name: count, dtype: int64
Bookings

status:
status
CONFIRMED    320
CANCELLED    314
PENDING      291
NaN           45
INVALID       30
Name: count, dtype: int64
Passengers

gender:
gender
M    545
F    494
Name: count, dtype: int64


In [15]:
# 4.7: Date/Time Inspection

print("Departure Time:")
display(flights_df["departure_time"].head(10))

print("\nArrival Time:")
display(flights_df["arrival_time"].head(10))

print("\nBooking Date:")
display(bookings_df["booking_date"].head(10))

print("\nDate of Birth:")
display(passengers_df["date_of_birth"].head(10))


# Checking for their actual types
print(flights_df[["departure_time", "arrival_time"]].dtypes)
print(bookings_df["booking_date"].dtype)
print(passengers_df["date_of_birth"].dtype)

Departure Time:


,departure_time
0,2026-04-20 23:38:41.701
1,2026-04-20 23:35:41.703
2,2026-04-20 23:26:41.702
3,2026-04-20 23:07:41.704
4,2026-04-20 23:05:41.703
5,2026-04-20 23:05:41.703
6,2026-04-20 23:04:41.703
7,2026-04-20 23:03:41.702
8,2026-04-20 23:02:41.701
9,2026-04-20 22:56:41.703



Arrival Time:


,arrival_time
0,2026-04-21 02:32:41.701
1,2026-04-21 01:23:41.703
2,2026-04-21 01:11:41.702
3,2026-04-21 01:43:41.704
4,2026-04-21 04:04:41.703
5,2026-04-21 01:24:41.703
6,2026-04-21 00:47:41.703
7,2026-04-21 00:35:41.702
8,2026-04-20 23:57:41.701
9,2026-04-20 23:37:41.703



Booking Date:


,booking_date
0,2025-06-14 11:37:36.951
1,2025-11-02 11:37:36.951
2,2025-08-25 11:37:36.951
3,2025-12-30 11:37:36.951
4,2025-10-02 11:37:36.951
5,2025-08-31 11:37:36.951
6,2025-06-29 11:37:36.951
7,2026-03-12 11:37:36.951
8,2025-09-07 11:37:36.951
9,2025-04-28 11:37:36.951



Date of Birth:


,date_of_birth
0,1974-04-08
1,2011-03-07
2,1954-09-10
3,1965-03-12
4,2005-11-11
5,1943-03-09
6,1939-02-25
7,1951-11-24
8,1951-11-10
9,1938-04-02


departure_time    datetime64[ns]
arrival_time      datetime64[ns]
dtype: object
datetime64[ns]
datetime64[ns]


In [16]:
# 4.8: Relationship/key profiling
# Booking to Flight - We are checking whether the existed flight_id in booking table is also presents in flight table's flight_id?
booking_flight_match = bookings_df["flight_id"].isin(flights_df["flight_id"])

print("Bookings with matching Flight ID:",
      booking_flight_match.sum())

print("Bookings with unmatched Flight ID:",
      (~booking_flight_match).sum())

# Booking to Passenger - We are checking whether the existed passenger_id in booking table is also presents in passenger table's passenger_id?
booking_passenger_match = bookings_df["passenger_id"].isin(passengers_df["passenger_id"])

print("Bookings with matching Passenger ID:",
      booking_passenger_match.sum())

print("Bookings with unmatched Passenger ID:",
      (~booking_passenger_match).sum())

# Payment to Booking - We are checking whether the existed booking_id in payment table is also presents in booking table's booking_id?
payment_booking_match = payments_df["booking_id"].isin(bookings_df["booking_id"])

print("Payments with matching Booking ID:",
      payment_booking_match.sum())

print("Payments with unmatched Booking ID:",
      (~payment_booking_match).sum())

Bookings with matching Flight ID: 1000
Bookings with unmatched Flight ID: 0
Bookings with matching Passenger ID: 1000
Bookings with unmatched Passenger ID: 0
Payments with matching Booking ID: 1000
Payments with unmatched Booking ID: 0


5. Data Quality Assessment

**5.1 Overview**

> The raw airline datasets were assessed for completeness, uniqueness, data-type consistency, validity, and referential integrity. The assessment focused on identifying missing values, duplicate records, inconsistent data types, potential flight-duration anomalies, and relationship issues between the four datasets. The identified issues are documented below and will be addressed during the data-cleaning and transformation stages.

**5.2 Identified Data Quality Issues**

| Dataset | Column | Quality Issue | Planned Treatment |
|---|---|---|---|
| Flights | `airline` | Missing values | Investigate and handle appropriately |
| Flights | Entire record | Duplicate records | Remove exact duplicates after verification |
| Payments | `amount` | Missing values and object/string datatype | Convert to numeric and handle missing values |
| Bookings | `status` | Missing values | Handle according to defined business rule |
| Passengers | `last_name` | Missing values | Preserve the passenger record and handle the missing attribute |
| Flights | `departure_time`, `arrival_time`, `duration` | Flight duration consistency requires validation | Calculate duration and compare with supplied duration |
| All relevant datasets | Key columns | Referential integrity requires validation | Identify and investigate unmatched keys |



**5.3 Duplicate Records**

> Exact duplicate records were identified in the Flights dataset. Duplicate records can result in double-counting during route, airline, and flight-level analysis. These records will therefore be investigated and exact duplicates removed during the cleaning stage.


**5.4 Missing Values**

> There are missing values found in various business-critical attributes. Missing values cannot be addressed based on a universal formula for all data sets. This will depend on the context and relevance of the attribute in question. All such decisions regarding missing values will be recorded under the data cleaning plan.



**5.5 Data-Type and Format Issues**

> The profiling stage identified fields requiring datatype and format standardization, particularly payment amounts and date/time-related fields. These fields will be converted into appropriate analytical data types during the cleaning and transformation stages.

**5.6 Flight Duration Validation**

> Flight duration will be independently calculated using departure and arrival times rather than relying solely on the supplied duration value. Overnight flights, where the arrival time occurs after midnight, will be handled by considering the arrival as occurring on the following calendar day. The calculated duration will then be compared with the provided duration to identify potential anomalies.

**5.7 Referential Integrity**

> Relationships between Flights, Bookings, Passengers, and Payments will be validated using their corresponding keys. Unmatched records will be identified and investigated before the datasets are integrated.

**5.8 Data Quality Objectives**

> The objectives of the cleansing and transformation processes include ensuring that the data generated is complete, without unintentional duplication, typed properly, consistent internally, referentially valid, and suitable for Power BI reporting. Personal identifying information will also be secured before creating the analytical dataset.

6. Data Cleaning

In [17]:
# Creating copies of raw datasets
# Because we are not performing clean operation directly on raw data

flights_clean = flights_df.copy()
payments_clean = payments_df.copy()
bookings_clean = bookings_df.copy()
passengers_clean = passengers_df.copy()

In [18]:
# 6.1: Remove Exact Duplicates
print("Flights shape before duplicate removal:", flights_clean.shape)
print("Duplicate rows before:", flights_clean.duplicated().sum())

# Remove exact duplicate rows
flights_clean = flights_clean.drop_duplicates().reset_index(drop=True)

print("\nFlights shape after duplicate removal:", flights_clean.shape)
print("Duplicate rows after:", flights_clean.duplicated().sum())

Flights shape before duplicate removal: (1020, 7)
Duplicate rows before: 15

Flights shape after duplicate removal: (1005, 7)
Duplicate rows after: 0


In [19]:
from re import S
# 6.2: Deriving Airlines from Flight ID
# While looking at flight_id, we can notice that the prefix of id contains the name of airline
# Such as SJ for SpiceJet, AI for Air India, UK for Vistara, 6F for IndiGo

# Make sure flight_id is treated as text
flights_clean["flight_id"] = (
    flights_clean["flight_id"]
    .astype("string")
    .str.strip()
)

# Extract the first TWO characters of the Flight ID
flights_clean["flight_prefix"] = (
    flights_clean["flight_id"]
    .str[:2]
    .str.upper()
)

# Define airline mapping
airline_mapping = {
    "SJ": "SpiceJet",
    "AI": "Air India",
    "UK": "Vistara",
    "6F": "IndiGo"
}

# Map prefix to airline
flights_clean["airline_from_id"] = (
    flights_clean["flight_prefix"]
    .map(airline_mapping)
)

# Check the results
print("Flight ID prefix distribution:")
print(flights_clean["flight_prefix"].value_counts(dropna=False))

print("\nDerived airline distribution:")
print(flights_clean["airline_from_id"].value_counts(dropna=False))

Flight ID prefix distribution:
flight_prefix
6F    273
AI    255
SJ    247
UK    230
Name: count, dtype: Int64

Derived airline distribution:
airline_from_id
IndiGo       273
Air India    255
SpiceJet     247
Vistara      230
Name: count, dtype: int64


In [20]:
# 6.3: Comparing Existing vs Derived

comparison = flights_clean[
    ["flight_id", "airline", "airline_from_id"]
].copy()

# Existing airline values that disagree with the Flight ID mapping
mismatches = comparison[
    comparison["airline"].notna() &
    comparison["airline_from_id"].notna() &
    (
        comparison["airline"].str.strip().str.lower()
        != comparison["airline_from_id"].str.strip().str.lower()
    )
]

print("Number of airline mismatches:", len(mismatches))

Number of airline mismatches: 30


In [21]:
# 6.4: Filling the airlines values using flight id

# Treat UNKNOWN as missing
flights_clean["airline"] = flights_clean["airline"].replace(
    ["UNKNOWN", "unknown", ""],
    pd.NA
)

# Fill missing airline ONLY when a valid Flight ID mapping exists
flights_clean["airline"] = (
    flights_clean["airline"]
    .fillna(flights_clean["airline_from_id"])
)

print("Airline values still missing:",
      flights_clean["airline"].isna().sum())

print("\nFinal airline distribution:")
print(flights_clean["airline"].value_counts(dropna=False))

Airline values still missing: 0

Final airline distribution:
airline
IndiGo       273
Air India    255
SpiceJet     247
Vistara      230
Name: count, dtype: int64


In [22]:
# 6.5: Cleaning Payment Amount

# amount column contains numeric values, missing values, and "INVALID"

print("Payment amount datatype before:")
print(payments_clean["amount"].dtype)

print("\nSample values before cleaning:")
print(payments_clean["amount"].head(10))

# Convert amount to numeric.
# Invalid values such as "INVALID" become NaN.
payments_clean["amount"] = pd.to_numeric(
    payments_clean["amount"],
    errors="coerce"
)

print("\nPayment amount datatype after:")
print(payments_clean["amount"].dtype)

print("\nMissing/invalid amounts after conversion:")
print(payments_clean["amount"].isna().sum())

Payment amount datatype before:
object

Sample values before cleaning:
0     9883.49
1     8457.96
2     6495.37
3     5079.38
4    12518.31
5     7319.71
6     8020.26
7      8291.6
8     7277.14
9     4585.24
Name: amount, dtype: object

Payment amount datatype after:
float64

Missing/invalid amounts after conversion:
78


In [23]:
# 6.6: Cleaning Booking Status
# For booking status it shows CONFIRMED, CANCELLED, PENDING, INVALID and missing values

print("Booking status before cleaning:")
print(bookings_clean["status"].value_counts(dropna=False))

# Remove leading/trailing spaces
bookings_clean["status"] = bookings_clean["status"].str.strip()

# Standardize invalid status
bookings_clean["status"] = bookings_clean["status"].replace(
    "INVALID", "Unknown"
)

# Standardize missing status
bookings_clean["status"] = bookings_clean["status"].fillna("Unknown")

print("\nBooking status after cleaning:")
print(bookings_clean["status"].value_counts(dropna=False))

Booking status before cleaning:
status
CONFIRMED    320
CANCELLED    314
PENDING      291
NaN           45
INVALID       30
Name: count, dtype: int64

Booking status after cleaning:
status
CONFIRMED    320
CANCELLED    314
PENDING      291
Unknown       75
Name: count, dtype: int64


In [24]:
# 6.7: Handling Missing Passenger Last Names

print("Missing last names before:",
      passengers_clean["last_name"].isna().sum())

# Replace missing last names
passengers_clean["last_name"] = passengers_clean["last_name"].fillna(
    "Unknown"
)

print("Missing last names after:",
      passengers_clean["last_name"].isna().sum())

Missing last names before: 10
Missing last names after: 0


In [25]:
# 6.8: Standardize Text Fields

# Flights
flight_text_columns = [
    "airline",
    "source",
    "destination"
]

# Payments
payment_text_columns = [
    "payment_method"
]

# Bookings
booking_text_columns = [
    "status"
]

# Passengers
passenger_text_columns = [
    "first_name",
    "last_name",
    "gender"
]

# Apply strip() to text columns
for column in flight_text_columns:
    flights_clean[column] = flights_clean[column].str.strip()

for column in payment_text_columns:
    payments_clean[column] = payments_clean[column].str.strip()

for column in booking_text_columns:
    bookings_clean[column] = bookings_clean[column].str.strip()

for column in passenger_text_columns:
    passengers_clean[column] = passengers_clean[column].str.strip()

print("Text standardization completed.")

Text standardization completed.


In [26]:
# 6.9: Check Identifier Quality
# We don't want to modify IDs, but we should make sure they remain unique.

print("Duplicate Flight IDs:",
      flights_clean["flight_id"].duplicated().sum())

print("Duplicate Payment IDs:",
      payments_clean["payment_id"].duplicated().sum())

print("Duplicate Booking IDs:",
      bookings_clean["booking_id"].duplicated().sum())

print("Duplicate Passenger IDs:",
      passengers_clean["passenger_id"].duplicated().sum())

Duplicate Flight IDs: 1
Duplicate Payment IDs: 0
Duplicate Booking IDs: 0
Duplicate Passenger IDs: 39


In [27]:
# 6.10: Final Dataset Shape

print("Flights shape after cleaning:", flights_clean.shape)
print("Payments shape after cleaning:", payments_clean.shape)
print("Bookings shape after cleaning:", bookings_clean.shape)
print("Passengers shape after cleaning:", passengers_clean.shape)

Flights shape after cleaning: (1005, 9)
Payments shape after cleaning: (1000, 4)
Bookings shape after cleaning: (1000, 9)
Passengers shape after cleaning: (1039, 9)


In [28]:
# 6.11: Preview of Cleaned Data

print("Cleaned Flights")
display(flights_clean.head(15))

print("\nCleaned Payments")
display(payments_clean.head())

print("\nCleaned Bookings")
display(bookings_clean.head())

print("Cleaned Passengers")
display(passengers_clean.head())

Cleaned Flights


,flight_id,airline,source,destination,departure_time,arrival_time,duration,flight_prefix,airline_from_id
0,SJ010,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,02:54:00,SJ,SpiceJet
1,AI155,Air India,BOM,CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,01:48:00,AI,Air India
2,UK094,Vistara,BOM,CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,01:45:00,UK,Vistara
3,AI245,Air India,BOM,CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,02:36:00,AI,Air India
4,AI192,Air India,MAA,BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,04:59:00,AI,Air India
5,SJ158,SpiceJet,DEL,HYD,2026-04-20 23:05:41.703,2026-04-21 01:24:41.703,02:19:00,SJ,SpiceJet
6,6F196,IndiGo,CCU,MAA,2026-04-20 23:04:41.703,2026-04-21 00:47:41.703,01:43:00,6F,IndiGo
7,AI080,Air India,BOM,HYD,2026-04-20 23:03:41.702,2026-04-21 00:35:41.702,01:32:00,AI,Air India
8,6F025,IndiGo,BLR,BOM,2026-04-20 23:02:41.701,2026-04-20 23:57:41.701,00:55:00,6F,IndiGo
9,6F251,IndiGo,DEL,BLR,2026-04-20 22:56:41.703,2026-04-20 23:37:41.703,00:41:00,6F,IndiGo



Cleaned Payments


,payment_id,booking_id,amount,payment_method
0,PAY1000,B1116,9883.49,NETBANKING
1,PAY1001,B1738,8457.96,NETBANKING
2,PAY1002,B1873,6495.37,UPI
3,PAY1003,B1914,5079.38,NETBANKING
4,PAY1004,B1967,12518.31,CARD



Cleaned Bookings


,booking_id,passenger_id,flight_id,booking_date,status,passport_number,seat_number,emergency_contact_name,emergency_contact_phone
0,B1000,P1591,AI192,2025-06-14 11:37:36.951,CANCELLED,P1945887,3D,Isaac Bakshi,+91-6478475128
1,B1001,P1803,6F026,2025-11-02 11:37:36.951,CANCELLED,L3482012,18A,Anvi Konda,+91-6647078662
2,B1002,P1083,SJ010,2025-08-25 11:37:36.951,CANCELLED,G8507659,30C,Udant Dewan,+91-8405938220
3,B1003,P1364,AI069,2025-12-30 11:37:36.951,CONFIRMED,M0891776,33A,Harsh Chahal,+91-6264636839
4,B1004,P1885,UK003,2025-10-02 11:37:36.951,PENDING,N5742231,25C,Pahal Balay,+91-9336478266


Cleaned Passengers


,passenger_id,first_name,last_name,age,gender,email,phone,aadhaar_id,date_of_birth
0,P1000,Vivaan,Chatterjee,52,F,vivaan.chatterjee@gmail.com,+91-6896233790,433218196001,1974-04-08
1,P1001,Krishna,Reddy,15,M,krishna.reddy@hotmail.com,+91-6702632297,386379402654,2011-03-07
2,P1002,Myra,Naidu,72,M,myra.naidu@outlook.com,+91-6199585092,615594078161,1954-09-10
3,P1003,Myra,Mishra,61,F,myra.mishra@hotmail.com,+91-8719927151,310341316475,1965-03-12
4,P1004,Saanvi,Banerjee,21,M,saanvi.banerjee@outlook.com,+91-7819595113,419283276483,2005-11-11


7. Flight Duration & Overnight Flight Handling

> Calculating the flight duration from departure and arrival timestamps, explicitly handle overnight flights, and comparing the calculated duration against the supplied duration.

In [29]:
# 7.1: Create a Working Copy

flights_transform = flights_clean.copy()
print("Flights dataset shape:", flights_transform.shape)

Flights dataset shape: (1005, 9)


In [30]:
# 7.2: Ensure Date/Time Columns Are Properly Formatted

flights_transform["departure_time"] = pd.to_datetime(
    flights_transform["departure_time"],
    errors="coerce"
)

flights_transform["arrival_time"] = pd.to_datetime(
    flights_transform["arrival_time"],
    errors="coerce"
)

print("Departure datatype:",
      flights_transform["departure_time"].dtype)

print("Arrival datatype:",
      flights_transform["arrival_time"].dtype)

print("\nMissing departure times:",
      flights_transform["departure_time"].isna().sum())

print("Missing arrival times:",
      flights_transform["arrival_time"].isna().sum())

Departure datatype: datetime64[ns]
Arrival datatype: datetime64[ns]

Missing departure times: 0
Missing arrival times: 0


In [31]:
# 7.3: Calculate Duration From Timestamps

flights_transform["calculated_duration"] = (
    flights_transform["arrival_time"]
    - flights_transform["departure_time"]
)

# Convert calculated duration into minutes
flights_transform["calculated_duration_minutes"] = (
    flights_transform["calculated_duration"]
    .dt.total_seconds()
    / 60
)

print(
    flights_transform[
        [
            "flight_id",
            "departure_time",
            "arrival_time",
            "calculated_duration",
            "calculated_duration_minutes"
        ]
    ].head(10)
)

  flight_id          departure_time            arrival_time  \
0     SJ010 2026-04-20 23:38:41.701 2026-04-21 02:32:41.701   
1     AI155 2026-04-20 23:35:41.703 2026-04-21 01:23:41.703   
2     UK094 2026-04-20 23:26:41.702 2026-04-21 01:11:41.702   
3     AI245 2026-04-20 23:07:41.704 2026-04-21 01:43:41.704   
4     AI192 2026-04-20 23:05:41.703 2026-04-21 04:04:41.703   
5     SJ158 2026-04-20 23:05:41.703 2026-04-21 01:24:41.703   
6     6F196 2026-04-20 23:04:41.703 2026-04-21 00:47:41.703   
7     AI080 2026-04-20 23:03:41.702 2026-04-21 00:35:41.702   
8     6F025 2026-04-20 23:02:41.701 2026-04-20 23:57:41.701   
9     6F251 2026-04-20 22:56:41.703 2026-04-20 23:37:41.703   

  calculated_duration  calculated_duration_minutes  
0     0 days 02:54:00                        174.0  
1     0 days 01:48:00                        108.0  
2     0 days 01:45:00                        105.0  
3     0 days 02:36:00                        156.0  
4     0 days 04:59:00                    

In [32]:
# 7.4: Identifying Overnight Flights

flights_transform["is_overnight"] = (
    flights_transform["arrival_time"].dt.date
    > flights_transform["departure_time"].dt.date
)

print("Overnight flight counts:")
print(
    flights_transform["is_overnight"]
    .value_counts(dropna=False)
)

Overnight flight counts:
is_overnight
False    883
True     122
Name: count, dtype: int64


In [33]:
# 7.5: Convert the Supplied Duration to Minutes

def time_to_minutes(value):
    """
    Convert a datetime.time value into total minutes.
    Returns NaN when the value cannot be converted.
    """

    if pd.isna(value):
        return np.nan

    try:
        return (
            value.hour * 60
            + value.minute
            + value.second / 60
        )
    except AttributeError:
        return np.nan


flights_transform["supplied_duration_minutes"] = (
    flights_transform["duration"]
    .apply(time_to_minutes)
)

print(
    flights_transform[
        [
            "flight_id",
            "duration",
            "supplied_duration_minutes"
        ]
    ].head(20)
)

   flight_id  duration  supplied_duration_minutes
0      SJ010  02:54:00                      174.0
1      AI155  01:48:00                      108.0
2      UK094  01:45:00                      105.0
3      AI245  02:36:00                      156.0
4      AI192  04:59:00                      299.0
5      SJ158  02:19:00                      139.0
6      6F196  01:43:00                      103.0
7      AI080  01:32:00                       92.0
8      6F025  00:55:00                       55.0
9      6F251  00:41:00                       41.0
10     AI137  02:28:00                      148.0
11     6F026  04:27:00                      267.0
12     UK167  03:01:00                      181.0
13     UK027  02:21:00                      141.0
14     AI069  03:46:00                      226.0
15     AI058  01:38:00                       98.0
16     AI140  03:43:00                      223.0
17     SJ239  01:46:00                      106.0
18     6F099  00:35:00                       35.0


In [34]:
# 7.6: Compare Calculated vs Supplied Duration

flights_transform["duration_difference_minutes"] = (
    flights_transform["calculated_duration_minutes"]
    - flights_transform["supplied_duration_minutes"]
)

display(
    flights_transform[
        [
            "flight_id",
            "supplied_duration_minutes",
            "calculated_duration_minutes",
            "duration_difference_minutes"
        ]
    ].head(20)
)

,flight_id,supplied_duration_minutes,calculated_duration_minutes,duration_difference_minutes
0,SJ010,174.0,174.0,0.0
1,AI155,108.0,108.0,0.0
2,UK094,105.0,105.0,0.0
3,AI245,156.0,156.0,0.0
4,AI192,299.0,299.0,0.0
5,SJ158,139.0,139.0,0.0
6,6F196,103.0,103.0,0.0
7,AI080,92.0,92.0,0.0
8,6F025,55.0,55.0,0.0
9,6F251,41.0,41.0,0.0


In [35]:
# 7.7: Define a Duration Anomaly Rule

DURATION_TOLERANCE_MINUTES = 5

flights_transform["duration_anomaly"] = (
    flights_transform["duration_difference_minutes"]
    .abs()
    > DURATION_TOLERANCE_MINUTES
)

print("Duration anomaly counts:")
print(
    flights_transform["duration_anomaly"]
    .value_counts(dropna=False)
)

Duration anomaly counts:
duration_anomaly
False    1004
True        1
Name: count, dtype: int64


In [36]:
# 7.8: Create an Anomaly Status

flights_transform["duration_status"] = np.select(
    [
        flights_transform["calculated_duration_minutes"].isna(),
        flights_transform["supplied_duration_minutes"].isna(),
        flights_transform["duration_anomaly"]
    ],
    [
        "Missing Timestamp",
        "Missing Supplied Duration",
        "Anomaly"
    ],
    default="Valid"
)

print(
    flights_transform["duration_status"]
    .value_counts(dropna=False)
)

duration_status
Valid      1004
Anomaly       1
Name: count, dtype: int64


In [37]:
# 7.9: Inspect Duration Anomalies

duration_anomalies = flights_transform[
    flights_transform["duration_status"] == "Anomaly"
][
    [
        "flight_id",
        "airline",
        "source",
        "destination",
        "departure_time",
        "arrival_time",
        "duration",
        "supplied_duration_minutes",
        "calculated_duration_minutes",
        "duration_difference_minutes",
        "is_overnight",
        "duration_status"
    ]
]

print("Number of duration anomalies:",
      len(duration_anomalies))

display(duration_anomalies.head(30))

Number of duration anomalies: 1


,flight_id,airline,source,destination,departure_time,arrival_time,duration,supplied_duration_minutes,calculated_duration_minutes,duration_difference_minutes,is_overnight,duration_status
351,SJ192,SpiceJet,HYD,BOM,2026-04-19 18:45:42,2026-04-18 23:45:42,1899-12-29 05:00:00,300.0,-1140.0,-1440.0,False,Anomaly


In [38]:
# 7.10: Add Duration in Hours

flights_transform["calculated_duration_hours"] = (
    flights_transform["calculated_duration_minutes"] / 60
)

flights_transform["supplied_duration_hours"] = (
    flights_transform["supplied_duration_minutes"] / 60
)

print(
    flights_transform[
        [
            "flight_id",
            "calculated_duration_minutes",
            "calculated_duration_hours"
        ]
    ].head(10)
)

  flight_id  calculated_duration_minutes  calculated_duration_hours
0     SJ010                        174.0                   2.900000
1     AI155                        108.0                   1.800000
2     UK094                        105.0                   1.750000
3     AI245                        156.0                   2.600000
4     AI192                        299.0                   4.983333
5     SJ158                        139.0                   2.316667
6     6F196                        103.0                   1.716667
7     AI080                         92.0                   1.533333
8     6F025                         55.0                   0.916667
9     6F251                         41.0                   0.683333


In [39]:
# 7.11: Create a Clean Analytical Duration Column

flights_transform["flight_duration_minutes"] = (
    flights_transform["calculated_duration_minutes"]
)

flights_transform["flight_duration_hours"] = (
    flights_transform["flight_duration_minutes"] / 60
)

In [40]:
# 7.12: Validate Overnight Flights

overnight_check = flights_transform[
    flights_transform["is_overnight"] == True
][
    [
        "flight_id",
        "departure_time",
        "arrival_time",
        "flight_duration_minutes"
    ]
]

print("Number of overnight flights:",
      len(overnight_check))

print("\nMinimum overnight duration:")
print(
    overnight_check["flight_duration_minutes"].min()
)

print("\nMaximum overnight duration:")
print(
    overnight_check["flight_duration_minutes"].max()
)

display(overnight_check.head(20))

Number of overnight flights: 122

Minimum overnight duration:
47.0

Maximum overnight duration:
299.0


,flight_id,departure_time,arrival_time,flight_duration_minutes
0,SJ010,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,174.0
1,AI155,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,108.0
2,UK094,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,105.0
3,AI245,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,156.0
4,AI192,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,299.0
5,SJ158,2026-04-20 23:05:41.703,2026-04-21 01:24:41.703,139.0
6,6F196,2026-04-20 23:04:41.703,2026-04-21 00:47:41.703,103.0
7,AI080,2026-04-20 23:03:41.702,2026-04-21 00:35:41.702,92.0
10,AI137,2026-04-20 22:49:41.702,2026-04-21 01:17:41.702,148.0
11,6F026,2026-04-20 22:46:42.000,2026-04-21 03:13:42.000,267.0


In [41]:
# 7.13 Check for Impossible/Negative Durations

negative_duration = flights_transform[
    flights_transform["flight_duration_minutes"] < 0
]

zero_duration = flights_transform[
    flights_transform["flight_duration_minutes"] == 0
]

print("Negative duration records:",
      len(negative_duration))

print("Zero duration records:",
      len(zero_duration))

Negative duration records: 1
Zero duration records: 0


In [42]:
# 7.14: Duration Summary

duration_summary = pd.DataFrame({
    "Metric": [
        "Total Flights",
        "Same-Day Flights",
        "Overnight Flights",
        "Duration Anomalies",
        "Missing Timestamp Records",
        "Missing Supplied Duration"
    ],
    "Count": [
        len(flights_transform),
        (flights_transform["is_overnight"] == False).sum(),
        (flights_transform["is_overnight"] == True).sum(),
        (flights_transform["duration_status"] == "Anomaly").sum(),
        (flights_transform["duration_status"] == "Missing Timestamp").sum(),
        (flights_transform["duration_status"] == "Missing Supplied Duration").sum()
    ]
})

display(duration_summary)

,Metric,Count
0,Total Flights,1005
1,Same-Day Flights,883
2,Overnight Flights,122
3,Duration Anomalies,1
4,Missing Timestamp Records,0
5,Missing Supplied Duration,0


In [43]:
# 7.15: Final Preview

display(
    flights_transform[
        [
            "flight_id",
            "airline",
            "source",
            "destination",
            "departure_time",
            "arrival_time",
            "flight_duration_minutes",
            "flight_duration_hours",
            "supplied_duration_minutes",
            "duration_difference_minutes",
            "duration_status"
        ]
    ].head(20)
)

,flight_id,airline,source,destination,departure_time,arrival_time,flight_duration_minutes,flight_duration_hours,supplied_duration_minutes,duration_difference_minutes,duration_status
0,SJ010,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,174.0,2.900000,174.0,0.0,Valid
1,AI155,Air India,BOM,CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,108.0,1.800000,108.0,0.0,Valid
2,UK094,Vistara,BOM,CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,105.0,1.750000,105.0,0.0,Valid
3,AI245,Air India,BOM,CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,156.0,2.600000,156.0,0.0,Valid
4,AI192,Air India,MAA,BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,299.0,4.983333,299.0,0.0,Valid
5,SJ158,SpiceJet,DEL,HYD,2026-04-20 23:05:41.703,2026-04-21 01:24:41.703,139.0,2.316667,139.0,0.0,Valid
6,6F196,IndiGo,CCU,MAA,2026-04-20 23:04:41.703,2026-04-21 00:47:41.703,103.0,1.716667,103.0,0.0,Valid
7,AI080,Air India,BOM,HYD,2026-04-20 23:03:41.702,2026-04-21 00:35:41.702,92.0,1.533333,92.0,0.0,Valid
8,6F025,IndiGo,BLR,BOM,2026-04-20 23:02:41.701,2026-04-20 23:57:41.701,55.0,0.916667,55.0,0.0,Valid
9,6F251,IndiGo,DEL,BLR,2026-04-20 22:56:41.703,2026-04-20 23:37:41.703,41.0,0.683333,41.0,0.0,Valid


**Section 7 Documentation**

> 7.16 Flight Duration and Overnight Flight Handling

Flight duration was independently calculated using the difference between the arrival timestamp and departure timestamp rather than relying solely on the supplied duration field.

The supplied duration was retained for data-quality comparison purposes.

Overnight flights were identified by comparing the calendar date of the arrival timestamp with the calendar date of the departure timestamp. Flights arriving on the following calendar day were classified as `Overnight`; otherwise, they were classified as `Same Day`.

A duration tolerance of 5 minutes was used when comparing calculated and supplied duration values. Differences greater than this threshold were flagged as duration anomalies.

Duration anomalies were retained rather than deleted so that they can be investigated and reported through the Power BI dashboard.

The calculated duration in minutes is used as the primary analytical duration metric.



8. PII Protection

In [45]:
# 8.1: Identify PII Columns

# Columns containing personally identifiable information
pii_columns = [
    "passport_number",
    "emergency_contact_name",
    "emergency_contact_phone"
]

# Passenger-level PII
passenger_pii_columns = [
    "first_name",
    "last_name",
    "email",
    "phone",
    "aadhaar_id",
    "date_of_birth"
]

print("Booking PII columns:")
print(pii_columns)

print("\nPassenger PII columns:")
print(passenger_pii_columns)

Booking PII columns:
['passport_number', 'emergency_contact_name', 'emergency_contact_phone']

Passenger PII columns:
['first_name', 'last_name', 'email', 'phone', 'aadhaar_id', 'date_of_birth']


In [48]:
# 8.2: Create Protected Passenger Dataset

passengers_protected = passengers_clean.copy()

print("Original passenger columns:")
print(passengers_protected.columns.tolist())

Original passenger columns:
['passenger_id', 'first_name', 'last_name', 'age', 'gender', 'email', 'phone', 'aadhaar_id', 'date_of_birth']


In [49]:
# 8.3: Hash Passenger ID

def hash_value(value):
    """
    Generate a SHA-256 hash for a value.
    """
    if pd.isna(value):
        return pd.NA

    return hashlib.sha256(
        str(value).encode("utf-8")
    ).hexdigest()


passengers_protected["passenger_id_hash"] = (
    passengers_protected["passenger_id"]
    .apply(hash_value)
)

print(
    passengers_protected[
        ["passenger_id", "passenger_id_hash"]
    ].head()
)

  passenger_id                                  passenger_id_hash
0        P1000  031a60e86398d41d18e79ffcedc168a3e137eec8e808ff...
1        P1001  61d0f276b7160f04f92ce7d852f04296612a23d85e1041...
2        P1002  c5baeafae225ad4d584ec322dd47d1bdde1c4e989d7169...
3        P1003  eeae6e0fb3e9e763d79dde942b8480b918448aefc3d538...
4        P1004  6533dc26f5dafb4abea1d39bc6d918a37c172db64bb11d...


In [50]:
# 8.4: Hash/Mask Sensitive Passenger Fields

def mask_value(value, visible_chars=2):
    """
    Mask a sensitive value while retaining a small
    portion for identification/debugging.
    """

    if pd.isna(value):
        return pd.NA

    value = str(value)

    if len(value) <= visible_chars:
        return "*" * len(value)

    return (
        value[:visible_chars]
        + "*" * (len(value) - visible_chars)
    )


# Aadhaar
passengers_protected["aadhaar_masked"] = (
    passengers_protected["aadhaar_id"]
    .apply(lambda x: mask_value(x, 2))
)

# Phone
passengers_protected["phone_masked"] = (
    passengers_protected["phone"]
    .apply(lambda x: mask_value(x, 2))
)

# Email
passengers_protected["email_masked"] = (
    passengers_protected["email"]
    .apply(lambda x: mask_value(x, 2))
)

In [52]:
# 8.5: Protect Booking PII

bookings_protected = bookings_clean.copy()

print("Original booking columns:")
print(bookings_protected.columns.tolist())

# Hashing Passenger ID in Bookings
bookings_protected["passenger_id_hash"] = (
    bookings_protected["passenger_id"]
    .apply(hash_value)
)

Original booking columns:
['booking_id', 'passenger_id', 'flight_id', 'booking_date', 'status', 'passport_number', 'seat_number', 'emergency_contact_name', 'emergency_contact_phone']


In [53]:
# 8.6: Mask Booking PII

bookings_protected["passport_number_masked"] = (
    bookings_protected["passport_number"]
    .apply(lambda x: mask_value(x, 2))
)

bookings_protected["emergency_contact_name_masked"] = (
    bookings_protected["emergency_contact_name"]
    .apply(lambda x: mask_value(x, 1))
)

bookings_protected["emergency_contact_phone_masked"] = (
    bookings_protected["emergency_contact_phone"]
    .apply(lambda x: mask_value(x, 2))
)

In [54]:
# 8.7: Remove Raw PII From BI Datasets
# The raw PII should not be included in the final Power BI dataset.

# Passenger BI dataset
passenger_bi = passengers_protected.drop(
    columns=[
        "passenger_id",
        "aadhaar_id",
        "phone",
        "email"
    ],
    errors="ignore"
)

# Booking BI dataset
booking_bi = bookings_protected.drop(
    columns=[
        "passenger_id",
        "passport_number",
        "emergency_contact_name",
        "emergency_contact_phone"
    ],
    errors="ignore"
)

print("Passenger BI columns:")
print(passenger_bi.columns.tolist())

print("\nBooking BI columns:")
print(booking_bi.columns.tolist())

Passenger BI columns:
['first_name', 'last_name', 'age', 'gender', 'date_of_birth', 'passenger_id_hash', 'aadhaar_masked', 'phone_masked', 'email_masked']

Booking BI columns:
['booking_id', 'flight_id', 'booking_date', 'status', 'seat_number', 'passenger_id_hash', 'passport_number_masked', 'emergency_contact_name_masked', 'emergency_contact_phone_masked']


In [55]:
# 8.8: Review Protected Data

print("Protected Passenger Data:")
display(passenger_bi.head())

print("\nProtected Booking Data:")
display(booking_bi.head())

Protected Passenger Data:


,first_name,last_name,age,gender,date_of_birth,passenger_id_hash,aadhaar_masked,phone_masked,email_masked
0,Vivaan,Chatterjee,52,F,1974-04-08,031a60e86398d41d18e79ffcedc168a3e137eec8e808ff...,43**********,+9************,vi*************************
1,Krishna,Reddy,15,M,2011-03-07,61d0f276b7160f04f92ce7d852f04296612a23d85e1041...,38**********,+9************,kr***********************
2,Myra,Naidu,72,M,1954-09-10,c5baeafae225ad4d584ec322dd47d1bdde1c4e989d7169...,61**********,+9************,my********************
3,Myra,Mishra,61,F,1965-03-12,eeae6e0fb3e9e763d79dde942b8480b918448aefc3d538...,31**********,+9************,my*********************
4,Saanvi,Banerjee,21,M,2005-11-11,6533dc26f5dafb4abea1d39bc6d918a37c172db64bb11d...,41**********,+9************,sa*************************



Protected Booking Data:


,booking_id,flight_id,booking_date,status,seat_number,passenger_id_hash,passport_number_masked,emergency_contact_name_masked,emergency_contact_phone_masked
0,B1000,AI192,2025-06-14 11:37:36.951,CANCELLED,3D,977c3154ed97450c76e731cfa64c1b4071ddd7001d11e1...,P1******,I***********,+9************
1,B1001,6F026,2025-11-02 11:37:36.951,CANCELLED,18A,604c9afd067fac61e2daa8a506a2030f2d8c48a0c123a8...,L3******,A*********,+9************
2,B1002,SJ010,2025-08-25 11:37:36.951,CANCELLED,30C,906ea2a505d51fd41a2b2f02d9dfc962d3831101cd8232...,G8******,U**********,+9************
3,B1003,AI069,2025-12-30 11:37:36.951,CONFIRMED,33A,253e3decf26bba87ebad66473c9ad56664e004ace4b3cf...,M0******,H***********,+9************
4,B1004,UK003,2025-10-02 11:37:36.951,PENDING,25C,f4133d1257c4df22d1f3160f167fa0b82a2d16ffc2fa60...,N5******,P**********,+9************


In [57]:
# 8.9: Verify Passenger ID Hash Consistency

# Take one passenger ID from the original dataset
sample_passenger_id = passengers_clean["passenger_id"].iloc[0]

# Hash it independently
sample_hash = hash_value(sample_passenger_id)

# Find it in the protected dataset
stored_hash = passengers_protected.loc[
    passengers_protected["passenger_id"] == sample_passenger_id,
    "passenger_id_hash"
].iloc[0]

print("Original Passenger ID:")
print(sample_passenger_id)

print("\nGenerated Hash:")
print(sample_hash)

print("\nStored Hash:")
print(stored_hash)

print("\nHash Match:")
print(sample_hash == stored_hash)

Original Passenger ID:
P1000

Generated Hash:
031a60e86398d41d18e79ffcedc168a3e137eec8e808ff3a26712c897d0c854a

Stored Hash:
031a60e86398d41d18e79ffcedc168a3e137eec8e808ff3a26712c897d0c854a

Hash Match:
True


In [58]:
# 8.10: Create a PII Protection Summary

pii_summary = pd.DataFrame({
    "Dataset": [
        "Passengers",
        "Passengers",
        "Passengers",
        "Bookings",
        "Bookings",
        "Bookings"
    ],
    "Sensitive Field": [
        "passenger_id",
        "aadhaar_id",
        "email / phone",
        "passenger_id",
        "passport_number",
        "emergency contact information"
    ],
    "Protection Method": [
        "SHA-256 hashing",
        "Masking",
        "Masking",
        "SHA-256 hashing",
        "Masking",
        "Masking"
    ]
})

display(pii_summary)

,Dataset,Sensitive Field,Protection Method
0,Passengers,passenger_id,SHA-256 hashing
1,Passengers,aadhaar_id,Masking
2,Passengers,email / phone,Masking
3,Bookings,passenger_id,SHA-256 hashing
4,Bookings,passport_number,Masking
5,Bookings,emergency contact information,Masking


**Section 8 Documentation**

> 8.11: PII Protection Strategy

The datasets contain personally identifiable information including passport numbers, Aadhaar IDs, email addresses, phone numbers, passenger names, and emergency-contact information.

To reduce exposure of sensitive information in the analytical layer, raw PII was excluded from the BI-ready datasets.

Passenger and booking identifiers used for relationships were transformed using SHA-256 hashing so that records can still be consistently linked without exposing the original identifier.

Sensitive attributes such as Aadhaar ID, passport number, email address, phone number, and emergency-contact information were masked or excluded from the BI layer.

Raw PII was not required for the analytical KPIs and therefore was not included in the final Power BI reporting dataset.

A validation check was implemented to ensure that raw sensitive fields were not present in the BI-ready passenger and booking datasets.

9. Data Validation

In [60]:
# 9.1: Create Validation Copies

validation_flights = flights_transform.copy()
validation_bookings = booking_bi.copy()
validation_payments = payments_clean.copy()
validation_passengers = passenger_bi.copy()

print("Validation datasets created.")

Validation datasets created.


In [61]:
# 9.2: Validate Flight IDs

valid_flight_id_pattern = r"^(SJ|AI|UK|6F)\d{3}$"

validation_flights["flight_id_valid"] = (
    validation_flights["flight_id"]
    .astype("string")
    .str.match(valid_flight_id_pattern, na=False)
)

print("Valid Flight IDs:")
print(
    validation_flights["flight_id_valid"]
    .value_counts()
)

Valid Flight IDs:
flight_id_valid
True    1005
Name: count, dtype: Int64


In [62]:
# 9.3: Inspect Malformed Flight IDs

malformed_flights = validation_flights[
    validation_flights["flight_id_valid"] == False
]

print("Number of malformed Flight IDs:",
      len(malformed_flights))

display(
    malformed_flights[
        [
            "flight_id",
            "airline",
            "source",
            "destination"
        ]
    ]
)

Number of malformed Flight IDs: 0


,flight_id,airline,source,destination


In [65]:
# 9.4: Validate Airline Against Flight ID

validation_flights["airline_expected"] = (
    validation_flights["flight_id"]
    .str[:2]
    .map(airline_mapping)
)

validation_flights["airline_match"] = (
    validation_flights["airline"]
    == validation_flights["airline_expected"]
)

print("Airline mapping validation:")
print(
    validation_flights["airline_match"]
    .value_counts(dropna=False)
)

Airline mapping validation:
airline_match
True    1005
Name: count, dtype: int64


In [66]:
# 9.5: Validate Flight Timestamps

validation_flights["departure_valid"] = (
    validation_flights["departure_time"].notna()
)

validation_flights["arrival_valid"] = (
    validation_flights["arrival_time"].notna()
)

print("Missing departure timestamps:",
      (~validation_flights["departure_valid"]).sum())

print("Missing arrival timestamps:",
      (~validation_flights["arrival_valid"]).sum())

Missing departure timestamps: 0
Missing arrival timestamps: 0


In [67]:
# 9.6: Validate Flight Duration

print("Negative calculated durations:",
      (
          validation_flights["flight_duration_minutes"] < 0
      ).sum()
)

print("Zero calculated durations:",
      (
          validation_flights["flight_duration_minutes"] == 0
      ).sum()
)

print("Duration anomalies:",
      (
          validation_flights["duration_status"] == "Anomaly"
      ).sum()
)

Negative calculated durations: 1
Zero calculated durations: 0
Duration anomalies: 1


In [68]:
# 9.7: Validate Booking IDs

print("Total bookings:",
      len(validation_bookings))

print("Duplicate booking IDs:",
      validation_bookings["booking_id"].duplicated().sum())

Total bookings: 1000
Duplicate booking IDs: 0


In [69]:
# 9.8: Validate Booking to Passenger Relationship

valid_passenger_hashes = set(
    passengers_protected["passenger_id_hash"]
    .dropna()
)

validation_bookings["passenger_exists"] = (
    validation_bookings["passenger_id_hash"]
    .isin(valid_passenger_hashes)
)

print("Bookings with valid passenger:",
      validation_bookings["passenger_exists"].sum())

print("Bookings with missing passenger:",
      (~validation_bookings["passenger_exists"]).sum())

Bookings with valid passenger: 1000
Bookings with missing passenger: 0


In [71]:
# 9.9: Validate Booking to Flight Relationship

valid_flight_ids = set(
    validation_flights["flight_id"]
    .dropna()
)

validation_bookings["flight_exists"] = (
    validation_bookings["flight_id"]
    .isin(valid_flight_ids)
)

print("Bookings with valid flight:",
      validation_bookings["flight_exists"].sum())

print("Bookings with missing flight:",
      (~validation_bookings["flight_exists"]).sum())

Bookings with valid flight: 1000
Bookings with missing flight: 0


In [72]:
# 9.10: Validate Payment to Booking Relationship

valid_booking_ids = set(
    validation_bookings["booking_id"]
    .dropna()
)

validation_payments["booking_exists"] = (
    validation_payments["booking_id"]
    .isin(valid_booking_ids)
)

print("Payments with valid booking:",
      validation_payments["booking_exists"].sum())

print("Payments with missing booking:",
      (~validation_payments["booking_exists"]).sum())

Payments with valid booking: 1000
Payments with missing booking: 0


In [73]:
# 9.11: Validate Payment Amounts

validation_payments["amount_valid"] = (
    validation_payments["amount"].notna()
    &
    (validation_payments["amount"] >= 0)
)

print("Valid payment amounts:",
      validation_payments["amount_valid"].sum())

print("Invalid/missing payment amounts:",
      (~validation_payments["amount_valid"]).sum())

Valid payment amounts: 922
Invalid/missing payment amounts: 78


In [74]:
# 9.12: Validate Booking Status

valid_statuses = {
    "CONFIRMED",
    "CANCELLED",
    "PENDING",
    "Unknown"
}

validation_bookings["status_valid"] = (
    validation_bookings["status"]
    .isin(valid_statuses)
)

print("Valid booking statuses:",
      validation_bookings["status_valid"].sum())

print("Invalid booking statuses:",
      (~validation_bookings["status_valid"]).sum())

Valid booking statuses: 1000
Invalid booking statuses: 0


In [75]:
# 9.13: PII Validation

raw_passenger_pii = [
    "aadhaar_id",
    "email",
    "phone"
]

raw_booking_pii = [
    "passport_number",
    "emergency_contact_name",
    "emergency_contact_phone"
]

passenger_pii_exposed = [
    col for col in raw_passenger_pii
    if col in validation_passengers.columns
]

booking_pii_exposed = [
    col for col in raw_booking_pii
    if col in validation_bookings.columns
]

print("Raw passenger PII exposed:")
print(passenger_pii_exposed)

print("\nRaw booking PII exposed:")
print(booking_pii_exposed)

Raw passenger PII exposed:
[]

Raw booking PII exposed:
[]


In [76]:
# 9.14: Overall Data Quality Report

data_quality_report = pd.DataFrame({
    "Validation Check": [
        "Duplicate flight rows",
        "Invalid flight IDs",
        "Airline mapping mismatches",
        "Missing departure timestamps",
        "Missing arrival timestamps",
        "Negative durations",
        "Duration anomalies",
        "Duplicate booking IDs",
        "Orphan passenger references",
        "Orphan flight references",
        "Orphan payment references",
        "Invalid/missing payment amounts",
        "Invalid booking statuses",
        "Raw passenger PII exposed",
        "Raw booking PII exposed"
    ],

    "Issue Count": [
        validation_flights.duplicated().sum(),
        (~validation_flights["flight_id_valid"]).sum(),
        (~validation_flights["airline_match"]).sum(),
        (~validation_flights["departure_valid"]).sum(),
        (~validation_flights["arrival_valid"]).sum(),
        (validation_flights["flight_duration_minutes"] < 0).sum(),
        (validation_flights["duration_status"] == "Anomaly").sum(),
        validation_bookings["booking_id"].duplicated().sum(),
        (~validation_bookings["passenger_exists"]).sum(),
        (~validation_bookings["flight_exists"]).sum(),
        (~validation_payments["booking_exists"]).sum(),
        (~validation_payments["amount_valid"]).sum(),
        (~validation_bookings["status_valid"]).sum(),
        len(passenger_pii_exposed),
        len(booking_pii_exposed)
    ]
})

display(data_quality_report)

,Validation Check,Issue Count
0,Duplicate flight rows,0
1,Invalid flight IDs,0
2,Airline mapping mismatches,0
3,Missing departure timestamps,0
4,Missing arrival timestamps,0
5,Negative durations,1
6,Duration anomalies,1
7,Duplicate booking IDs,0
8,Orphan passenger references,0
9,Orphan flight references,0


In [79]:
# 9.15: Overall Validation Status

critical_checks = {
    "Invalid Flight IDs": (~validation_flights["flight_id_valid"]).sum(),
    "Airline mismatches": (~validation_flights["airline_match"]).sum(),
    "Negative durations": (
        validation_flights["flight_duration_minutes"] < 0
    ).sum(),
    "Orphan passenger references": (
        ~validation_bookings["passenger_exists"]
    ).sum(),
    "Orphan flight references": (
        ~validation_bookings["flight_exists"]
    ).sum(),
    "Orphan payment references": (
        ~validation_payments["booking_exists"]
    ).sum(),
    "Raw PII exposure": (
        len(passenger_pii_exposed) +
        len(booking_pii_exposed)
    )
}

print("Overall Data Validation!!!\n")

for check, count in critical_checks.items():
    status = "PASS" if count == 0 else "REVIEW"
    print(f"{check}: {count} -> {status}")

Overall Data Validation!!!

Invalid Flight IDs: 0 -> PASS
Airline mismatches: 0 -> PASS
Negative durations: 1 -> REVIEW
Orphan passenger references: 0 -> PASS
Orphan flight references: 0 -> PASS
Orphan payment references: 0 -> PASS
Raw PII exposure: 0 -> PASS


**Section 9 Documentation**

> 9.16: Data Validation Strategy

The transformed datasets were subjected to multiple validation checks before analytical integration.

Validation included:

- Flight ID format validation
- Airline consistency validation using the Flight ID business rule
- Duplicate identifier checks
- Timestamp completeness checks
- Flight duration validation
- Duration anomaly detection
- Booking-to-passenger referential integrity
- Booking-to-flight referential integrity
- Payment-to-booking referential integrity
- Payment amount validation
- Booking status validation
- PII exposure validation

Records identified as anomalies were retained where possible and flagged for analytical investigation rather than being silently removed.

The validation layer ensures that the downstream analytical dataset is suitable for KPI calculation and Power BI reporting.

10. Data Integration

In [80]:
# 10.1: Clean/Protected Tables

# Use the transformed flights table
flights_final = flights_transform.copy()

# Use protected passenger and booking tables
passengers_final = passenger_bi.copy()
bookings_final = booking_bi.copy()

# Payments
payments_final = payments_clean.copy()

print("Flights:", flights_final.shape)
print("Bookings:", bookings_final.shape)
print("Passengers:", passengers_final.shape)
print("Payments:", payments_final.shape)

Flights: (1005, 20)
Bookings: (1000, 9)
Passengers: (1039, 9)
Payments: (1000, 4)


In [81]:
# 10.2: Preparing Flights Table

temporary_flight_columns = [
    "flight_prefix",
    "airline_from_id",
    "airline_expected",
    "airline_match",
    "flight_id_valid",
    "departure_valid",
    "arrival_valid"
]

flights_final = flights_final.drop(
    columns=temporary_flight_columns,
    errors="ignore"
)

print("Final Flights columns:")
print(flights_final.columns.tolist())

Final Flights columns:
['flight_id', 'airline', 'source', 'destination', 'departure_time', 'arrival_time', 'duration', 'calculated_duration', 'calculated_duration_minutes', 'is_overnight', 'supplied_duration_minutes', 'duration_difference_minutes', 'duration_anomaly', 'duration_status', 'calculated_duration_hours', 'supplied_duration_hours', 'flight_duration_minutes', 'flight_duration_hours']


In [82]:
# 10.3: Prepare Bookings Table

temporary_booking_columns = [
    "passenger_exists",
    "flight_exists",
    "status_valid"
]

bookings_final = bookings_final.drop(
    columns=temporary_booking_columns,
    errors="ignore"
)

print("Final Bookings columns:")
print(bookings_final.columns.tolist())

Final Bookings columns:
['booking_id', 'flight_id', 'booking_date', 'status', 'seat_number', 'passenger_id_hash', 'passport_number_masked', 'emergency_contact_name_masked', 'emergency_contact_phone_masked']


In [83]:
# 10.4: Prepare Payments Table

payments_final = payments_final.drop(
    columns=["booking_exists", "amount_valid"],
    errors="ignore"
)

print("Final Payments columns:")
print(payments_final.columns.tolist())

Final Payments columns:
['payment_id', 'booking_id', 'amount', 'payment_method']


In [84]:
# 10.5: Merge Bookings With Flights

booking_flight = bookings_final.merge(
    flights_final,
    on="flight_id",
    how="left",
    suffixes=("_booking", "_flight"),
    indicator=True
)

print("Booking + Flight shape:")
print(booking_flight.shape)

print("\nMerge status:")
print(booking_flight["_merge"].value_counts())

Booking + Flight shape:
(1002, 27)

Merge status:
_merge
both          1002
left_only        0
right_only       0
Name: count, dtype: int64


In [85]:
# 10.6: Check Unmatched Bookings

unmatched_bookings_flights = booking_flight[
    booking_flight["_merge"] == "left_only"
]

print(
    "Bookings without matching flight:",
    len(unmatched_bookings_flights)
)

display(
    unmatched_bookings_flights[
        ["booking_id", "flight_id"]
    ].head(20)
)

Bookings without matching flight: 0


,booking_id,flight_id


In [86]:
# 10.7: Remove Merge Indicator

booking_flight = booking_flight.drop(
    columns=["_merge"]
)

In [87]:
# 10.8: Merge With Payments

analytical_df = booking_flight.merge(
    payments_final,
    on="booking_id",
    how="left",
    suffixes=("", "_payment"),
    indicator=True
)

print("Final analytical dataset shape:")
print(analytical_df.shape)

print("\nPayment merge status:")
print(analytical_df["_merge"].value_counts())

Final analytical dataset shape:
(1366, 30)

Payment merge status:
_merge
both          1002
left_only      364
right_only       0
Name: count, dtype: int64


In [88]:
# 10.9: Check Bookings Without Payments

unmatched_payments = analytical_df[
    analytical_df["_merge"] == "left_only"
]

print(
    "Bookings without matching payment:",
    len(unmatched_payments)
)

display(
    unmatched_payments[
        ["booking_id"]
    ].head(20)
)

Bookings without matching payment: 364


,booking_id
0,B1000
1,B1001
4,B1003
8,B1007
9,B1008
10,B1009
18,B1014
23,B1017
30,B1022
39,B1028


In [89]:
# 10.10: Remove Merge Indicator

analytical_df = analytical_df.drop(
    columns=["_merge"]
)

In [90]:
# 10.11: Merge Passenger Information

analytical_df = analytical_df.merge(
    passengers_final,
    on="passenger_id_hash",
    how="left",
    suffixes=("", "_passenger"),
    indicator=True
)

print("After passenger merge:")
print(analytical_df.shape)

print("\nPassenger merge status:")
print(analytical_df["_merge"].value_counts())

After passenger merge:
(1420, 38)

Passenger merge status:
_merge
both          1420
left_only        0
right_only       0
Name: count, dtype: int64


In [91]:
# 10.12: Check Unmatched Passengers

unmatched_passengers = analytical_df[
    analytical_df["_merge"] == "left_only"
]

print(
    "Bookings without matching passenger:",
    len(unmatched_passengers)
)

display(
    unmatched_passengers[
        ["booking_id", "passenger_id_hash"]
    ].head(20)
)

Bookings without matching passenger: 0


,booking_id,passenger_id_hash


In [92]:
# 10.13: Remove Merge Indicator

analytical_df = analytical_df.drop(
    columns=["_merge"]
)

In [93]:
# 10.14: Inspect the Integrated Dataset

print("Integrated dataset shape:")
print(analytical_df.shape)

print("\nColumns:")
print(analytical_df.columns.tolist())

display(analytical_df.head())

Integrated dataset shape:
(1420, 37)

Columns:
['booking_id', 'flight_id', 'booking_date', 'status', 'seat_number', 'passenger_id_hash', 'passport_number_masked', 'emergency_contact_name_masked', 'emergency_contact_phone_masked', 'airline', 'source', 'destination', 'departure_time', 'arrival_time', 'duration', 'calculated_duration', 'calculated_duration_minutes', 'is_overnight', 'supplied_duration_minutes', 'duration_difference_minutes', 'duration_anomaly', 'duration_status', 'calculated_duration_hours', 'supplied_duration_hours', 'flight_duration_minutes', 'flight_duration_hours', 'payment_id', 'amount', 'payment_method', 'first_name', 'last_name', 'age', 'gender', 'date_of_birth', 'aadhaar_masked', 'phone_masked', 'email_masked']


,booking_id,flight_id,booking_date,status,seat_number,passenger_id_hash,passport_number_masked,emergency_contact_name_masked,emergency_contact_phone_masked,airline,...,amount,payment_method,first_name,last_name,age,gender,date_of_birth,aadhaar_masked,phone_masked,email_masked
0,B1000,AI192,2025-06-14 11:37:36.951,CANCELLED,3D,977c3154ed97450c76e731cfa64c1b4071ddd7001d11e1...,P1******,I***********,+9************,Air India,...,NaN,NaN,Krishna,Mehta,4,M,2022-05-23,55*********,+9************,kr**************************
1,B1001,6F026,2025-11-02 11:37:36.951,CANCELLED,18A,604c9afd067fac61e2daa8a506a2030f2d8c48a0c123a8...,L3******,A*********,+9************,IndiGo,...,NaN,NaN,Krishna,Singh,82,F,1944-01-03,28*********,+9************,kr***********************
2,B1002,SJ010,2025-08-25 11:37:36.951,CANCELLED,30C,906ea2a505d51fd41a2b2f02d9dfc962d3831101cd8232...,G8******,U**********,+9************,SpiceJet,...,6087.22,UPI,Arjun,Banerjee,62,F,1964-02-24,44**********,+9************,ar************************
3,B1002,SJ010,2025-08-25 11:37:36.951,CANCELLED,30C,906ea2a505d51fd41a2b2f02d9dfc962d3831101cd8232...,G8******,U**********,+9************,SpiceJet,...,2505.86,NETBANKING,Arjun,Banerjee,62,F,1964-02-24,44**********,+9************,ar************************
4,B1003,AI069,2025-12-30 11:37:36.951,CONFIRMED,33A,253e3decf26bba87ebad66473c9ad56664e004ace4b3cf...,M0******,H***********,+9************,Air India,...,NaN,NaN,Aarav,Ghosh,60,M,1966-12-21,95**********,+9************,aa**********************


In [95]:
# 10.15: Check Final Record Count

print("Number of bookings:", len(bookings_final))
print("Number of analytical records:", len(analytical_df))

Number of bookings: 1000
Number of analytical records: 1420


In [106]:
# 10.16: Select BI Columns

analycal_columns = [
    # Booking information
    "booking_id",
    "passenger_id_hash",
    "flight_id",
    "booking_date",
    "status",
    "seat_number",

    # Flight information
    "airline",
    "source",
    "destination",
    "departure_time",
    "arrival_time",
    "is_overnight",  # Include 'is_overnight' for 'flight_day_type' calculation
    "flight_duration_minutes",
    "flight_duration_hours",
    "supplied_duration_minutes",
    "duration_difference_minutes",
    "duration_status",

    # Payment information
    "payment_id",
    "amount",
    "payment_method",

    # Passenger information
    "first_name",
    "last_name",
    "age",
    "gender",
    "aadhaar_masked",
    "phone_masked",
    "email_masked"
]

# Keep only columns that actually exist
analytical_columns = [
    col for col in analytical_columns
    if col in analytical_df.columns
]

analycal_bi = analytical_df[analytical_columns].copy()

print("BI-ready analytical dataset:")
print(analytical_bi.shape)

display(analytical_bi.head())

BI-ready analytical dataset:
(1420, 26)


,booking_id,passenger_id_hash,flight_id,booking_date,status,seat_number,airline,source,destination,departure_time,...,payment_id,amount,payment_method,first_name,last_name,age,gender,aadhaar_masked,phone_masked,email_masked
0,B1000,977c3154ed97450c76e731cfa64c1b4071ddd7001d11e1...,AI192,2025-06-14 11:37:36.951,CANCELLED,3D,Air India,MAA,BOM,2026-04-20 23:05:41.703,...,NaN,NaN,NaN,Krishna,Mehta,4,M,55*********,+9************,kr**************************
1,B1001,604c9afd067fac61e2daa8a506a2030f2d8c48a0c123a8...,6F026,2025-11-02 11:37:36.951,CANCELLED,18A,IndiGo,BOM,CCU,2026-04-20 22:46:42.000,...,NaN,NaN,NaN,Krishna,Singh,82,F,28*********,+9************,kr***********************
2,B1002,906ea2a505d51fd41a2b2f02d9dfc962d3831101cd8232...,SJ010,2025-08-25 11:37:36.951,CANCELLED,30C,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,...,PAY1305,6087.22,UPI,Arjun,Banerjee,62,F,44**********,+9************,ar************************
3,B1002,906ea2a505d51fd41a2b2f02d9dfc962d3831101cd8232...,SJ010,2025-08-25 11:37:36.951,CANCELLED,30C,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,...,PAY1912,2505.86,NETBANKING,Arjun,Banerjee,62,F,44**********,+9************,ar************************
4,B1003,253e3decf26bba87ebad66473c9ad56664e004ace4b3cf...,AI069,2025-12-30 11:37:36.951,CONFIRMED,33A,Air India,DEL,BOM,2026-04-20 22:43:41.702,...,NaN,NaN,NaN,Aarav,Ghosh,60,M,95**********,+9************,aa**********************


In [97]:
# 10.17: Final PII Check

raw_pii_columns = [
    "passport_number",
    "emergency_contact_name",
    "emergency_contact_phone",
    "aadhaar_id",
    "email",
    "phone"
]

exposed_pii = [
    col for col in raw_pii_columns
    if col in analytical_bi.columns
]

print("Raw PII fields present in analytical dataset:")
print(exposed_pii)

Raw PII fields present in analytical dataset:
[]


In [99]:
# 10.18: Final Data Quality Check

print("Final Analytical Data Quality Check!!\n")
print("\nRows:", len(analytical_bi))
print("Columns:", len(analytical_bi.columns))

print(
    "Duplicate booking IDs:",
    analytical_bi["booking_id"].duplicated().sum()
)

print(
    "Missing flight IDs:",
    analytical_bi["flight_id"].isna().sum()
)

print(
    "Missing airline:",
    analytical_bi["airline"].isna().sum()
)

print(
    "Missing passenger hash:",
    analytical_bi["passenger_id_hash"].isna().sum()
)

print(
    "Raw PII exposed:",
    len(exposed_pii)
)

Final Analytical Data Quality Check!!


Rows: 1420
Columns: 26
Duplicate booking IDs: 420
Missing flight IDs: 0
Missing airline: 0
Missing passenger hash: 0
Raw PII exposed: 0


In [101]:
# 10.19: Saving the Integrated Dataset

from pathlib import Path

output_dir = Path("/content/output")
output_dir.mkdir(parents=True, exist_ok=True)

print("Output directory:", output_dir)

analytical_bi.to_csv(
    output_dir / "airline_analytical_data.csv",
    index=False
)

print("Analytical dataset exported successfully.")

Output directory: /content/output
Analytical dataset exported successfully.


In [102]:
# 10.20: Exporting Individual Clean Tables

flights_final.to_csv(
    output_dir / "flights_clean.csv",
    index=False
)

bookings_final.to_csv(
    output_dir / "bookings_protected.csv",
    index=False
)

payments_final.to_csv(
    output_dir / "payments_clean.csv",
    index=False
)

passengers_final.to_csv(
    output_dir / "passengers_protected.csv",
    index=False
)

analytical_bi.to_csv(
    output_dir / "airline_analytical_data.csv",
    index=False
)

print("All processed datasets exported.")

All processed datasets exported.


11. KPIs & Aggregation

In [103]:
# 11.1: Create a KPI Summary

# Work from the final BI-ready analytical dataset
kpi_df = analytical_bi.copy()

print("Analytical dataset shape:", kpi_df.shape)

Analytical dataset shape: (1420, 26)


In [105]:
# 11.2: Overall KPI Metrics

# Create 'flight_day_type' from 'is_overnight' column if it exists
if 'is_overnight' in kpi_df.columns:
    kpi_df['flight_day_type'] = kpi_df['is_overnight'].apply(
        lambda x: 'Overnight' if x else 'Same Day'
    )
else:
    # If 'is_overnight' is not found, handle this case (e.g., set a default or log a warning)
    # For now, we'll set it to 'Unknown' or similar if 'is_overnight' was never created
    kpi_df['flight_day_type'] = 'Unknown'
    print("Warning: 'is_overnight' column not found, 'flight_day_type' set to 'Unknown'.")

total_bookings = kpi_df["booking_id"].nunique()

total_flights = kpi_df["flight_id"].nunique()

total_airlines = kpi_df["airline"].nunique()

average_flight_duration = (
    kpi_df["flight_duration_minutes"].mean()
)

overnight_flights = (
    kpi_df["flight_day_type"] == "Overnight"
).sum()

duration_anomalies = (
    kpi_df["duration_status"] == "Anomaly"
).sum()

total_revenue = kpi_df["amount"].sum()

average_payment = kpi_df["amount"].mean()

confirmed_bookings = (
    kpi_df["status"] == "CONFIRMED"
).sum()

cancelled_bookings = (
    kpi_df["status"] == "CANCELLED"
).sum()

pending_bookings = (
    kpi_df["status"] == "PENDING"
).sum()

print("Total Bookings:", total_bookings)
print("Total Flights:", total_flights)
print("Total Airlines:", total_airlines)
print("Average Flight Duration (minutes):", average_flight_duration)
print("Overnight Flights:", overnight_flights)
print("Duration Anomalies:", duration_anomalies)
print("Total Payment Amount:", total_revenue)
print("Average Payment:", average_payment)
print("Confirmed Bookings:", confirmed_bookings)
print("Cancelled Bookings:", cancelled_bookings)
print("Pending Bookings:", pending_bookings)

Total Bookings: 1000
Total Flights: 984
Total Airlines: 4
Average Flight Duration (minutes): 161.3438054225352
Overnight Flights: 0
Duration Anomalies: 1
Total Payment Amount: 7749528.26
Average Payment: 8022.285983436853
Confirmed Bookings: 461
Cancelled Bookings: 434
Pending Bookings: 428


In [107]:
# 11.3: Create KPI Summary Table

kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Bookings",
        "Total Flights",
        "Total Airlines",
        "Average Flight Duration (Minutes)",
        "Overnight Flights",
        "Duration Anomalies",
        "Total Payment Amount",
        "Average Payment Amount",
        "Confirmed Bookings",
        "Cancelled Bookings",
        "Pending Bookings"
    ],
    "Value": [
        total_bookings,
        total_flights,
        total_airlines,
        round(average_flight_duration, 2),
        overnight_flights,
        duration_anomalies,
        round(total_revenue, 2),
        round(average_payment, 2),
        confirmed_bookings,
        cancelled_bookings,
        pending_bookings
    ]
})

display(kpi_summary)

,KPI,Value
0,Total Bookings,1000.00
1,Total Flights,984.00
2,Total Airlines,4.00
3,Average Flight Duration (Minutes),161.34
4,Overnight Flights,0.00
5,Duration Anomalies,1.00
6,Total Payment Amount,7749528.26
7,Average Payment Amount,8022.29
8,Confirmed Bookings,461.00
9,Cancelled Bookings,434.00


In [108]:
# 11.4: Airline-wise Distribution

airline_summary = (
    kpi_df
    .groupby("airline", dropna=False)
    .agg(
        bookings=("booking_id", "nunique"),
        flights=("flight_id", "nunique"),
        average_duration_minutes=(
            "flight_duration_minutes",
            "mean"
        ),
        duration_anomalies=(
            "duration_status",
            lambda x: (x == "Anomaly").sum()
        ),
        total_payment=("amount", "sum")
    )
    .reset_index()
)

airline_summary["booking_share_percent"] = (
    airline_summary["bookings"]
    / airline_summary["bookings"].sum()
    * 100
)

airline_summary["average_duration_hours"] = (
    airline_summary["average_duration_minutes"] / 60
)

display(airline_summary)

,airline,bookings,flights,average_duration_minutes,duration_anomalies,total_payment,booking_share_percent,average_duration_hours
0,Air India,251,246,163.795731,0,1859548.48,25.1,2.729929
1,IndiGo,270,269,163.150556,0,1692911.17,27.0,2.719176
2,SpiceJet,248,244,154.444231,1,2021745.57,24.8,2.574071
3,Vistara,231,225,164.082301,0,2175323.04,23.1,2.734705


In [109]:
# 11.5: Route-wise Traffic

route_summary = (
    kpi_df
    .groupby(
        ["source", "destination"],
        dropna=False
    )
    .agg(
        bookings=("booking_id", "nunique"),
        flights=("flight_id", "nunique"),
        average_duration_minutes=(
            "flight_duration_minutes",
            "mean"
        ),
        duration_anomalies=(
            "duration_status",
            lambda x: (x == "Anomaly").sum()
        )
    )
    .reset_index()
)

# Create a readable route column
route_summary["route"] = (
    route_summary["source"]
    + " → "
    + route_summary["destination"]
)

# Sort by traffic
route_summary = route_summary.sort_values(
    "bookings",
    ascending=False
).reset_index(drop=True)

display(route_summary.head(20))

,source,destination,bookings,flights,average_duration_minutes,duration_anomalies,route
0,BOM,CCU,87,87,164.243983,0,BOM → CCU
1,CCU,DEL,73,71,151.158515,0,CCU → DEL
2,MAA,BLR,64,64,162.191648,0,MAA → BLR
3,BLR,BOM,62,60,146.494784,0,BLR → BOM
4,HYD,MAA,57,57,141.506204,0,HYD → MAA
5,DEL,HYD,55,54,179.590361,0,DEL → HYD
6,HYD,DEL,41,41,186.666850,0,HYD → DEL
7,BOM,DEL,36,36,149.927362,0,BOM → DEL
8,CCU,BOM,34,33,157.355996,0,CCU → BOM
9,DEL,BLR,30,29,172.704884,0,DEL → BLR


In [110]:
# 11.6: Route Traffic Percentage

route_summary["traffic_share_percent"] = (
    route_summary["bookings"]
    / route_summary["bookings"].sum()
    * 100
)

display(route_summary.head(20))

,source,destination,bookings,flights,average_duration_minutes,duration_anomalies,route,traffic_share_percent
0,BOM,CCU,87,87,164.243983,0,BOM → CCU,8.682635
1,CCU,DEL,73,71,151.158515,0,CCU → DEL,7.285429
2,MAA,BLR,64,64,162.191648,0,MAA → BLR,6.387226
3,BLR,BOM,62,60,146.494784,0,BLR → BOM,6.187625
4,HYD,MAA,57,57,141.506204,0,HYD → MAA,5.688623
5,DEL,HYD,55,54,179.590361,0,DEL → HYD,5.489022
6,HYD,DEL,41,41,186.666850,0,HYD → DEL,4.091816
7,BOM,DEL,36,36,149.927362,0,BOM → DEL,3.592814
8,CCU,BOM,34,33,157.355996,0,CCU → BOM,3.393214
9,DEL,BLR,30,29,172.704884,0,DEL → BLR,2.994012


In [111]:
# 11.7: Duration Analysis

duration_summary_by_airline = (
    kpi_df
    .groupby("airline")
    .agg(
        average_duration_minutes=(
            "flight_duration_minutes",
            "mean"
        ),
        minimum_duration_minutes=(
            "flight_duration_minutes",
            "min"
        ),
        maximum_duration_minutes=(
            "flight_duration_minutes",
            "max"
        ),
        flight_count=(
            "flight_id",
            "nunique"
        )
    )
    .reset_index()
)

duration_summary_by_airline[
    "average_duration_hours"
] = (
    duration_summary_by_airline[
        "average_duration_minutes"
    ] / 60
)

display(duration_summary_by_airline)

,airline,average_duration_minutes,minimum_duration_minutes,maximum_duration_minutes,flight_count,average_duration_hours
0,Air India,163.795731,35.0,299.0,246,2.729929
1,IndiGo,163.150556,30.0,300.0,269,2.719176
2,SpiceJet,154.444231,-1140.0,300.0,244,2.574071
3,Vistara,164.082301,30.0,300.0,225,2.734705


In [112]:
# 11.8: Overnight Flight Analysis

overnight_summary = (
    kpi_df
    .groupby("flight_day_type")
    .agg(
        bookings=("booking_id", "nunique"),
        flights=("flight_id", "nunique"),
        average_duration_minutes=(
            "flight_duration_minutes",
            "mean"
        )
    )
    .reset_index()
)

display(overnight_summary)

,flight_day_type,bookings,flights,average_duration_minutes
0,Unknown,1000,984,161.343805


In [113]:
# 11.9: Duration Anomaly Analysis

anomaly_summary = (
    kpi_df
    .groupby("duration_status")
    .agg(
        records=("booking_id", "nunique"),
        average_duration_minutes=(
            "flight_duration_minutes",
            "mean"
        )
    )
    .reset_index()
)

display(anomaly_summary)

,duration_status,records,average_duration_minutes
0,Anomaly,1,-1140.000000
1,Valid,999,162.260891


In [114]:
# 11.10: Airline Anomaly Rate

airline_anomaly = (
    kpi_df
    .groupby("airline")
    .agg(
        total_records=("booking_id", "nunique"),
        anomalies=(
            "duration_status",
            lambda x: (x == "Anomaly").sum()
        )
    )
    .reset_index()
)

airline_anomaly["anomaly_rate_percent"] = (
    airline_anomaly["anomalies"]
    / airline_anomaly["total_records"]
    * 100
)

display(airline_anomaly)

,airline,total_records,anomalies,anomaly_rate_percent
0,Air India,251,0,0.000000
1,IndiGo,270,0,0.000000
2,SpiceJet,248,1,0.403226
3,Vistara,231,0,0.000000


In [115]:
# 11.11: Booking Status Analysis

booking_status_summary = (
    kpi_df
    .groupby("status")
    .agg(
        bookings=("booking_id", "nunique"),
        total_payment=("amount", "sum")
    )
    .reset_index()
)

booking_status_summary["booking_share_percent"] = (
    booking_status_summary["bookings"]
    / booking_status_summary["bookings"].sum()
    * 100
)

display(booking_status_summary)

,status,bookings,total_payment,booking_share_percent
0,CANCELLED,314,2313885.71,31.4
1,CONFIRMED,320,2643413.79,32.0
2,PENDING,291,2381130.53,29.1
3,Unknown,75,411098.23,7.5


In [116]:
# 11.12: Age Group Analysis

kpi_df["age_group"] = pd.cut(
    kpi_df["age"],
    bins=[0, 18, 30, 45, 60, 120],
    labels=[
        "Under 18",
        "18-30",
        "31-45",
        "46-60",
        "60+"
    ],
    include_lowest=True
)

age_summary = (
    kpi_df
    .groupby("age_group", observed=False)
    .agg(
        passengers=("passenger_id_hash", "nunique"),
        bookings=("booking_id", "nunique")
    )
    .reset_index()
)

display(age_summary)

,age_group,passengers,bookings
0,Under 18,134,220
1,18-30,91,146
2,31-45,98,155
3,46-60,123,181
4,60+,190,298


In [117]:
# 11.13: Monthly Booking Trend

kpi_df["booking_date"] = pd.to_datetime(
    kpi_df["booking_date"],
    errors="coerce"
)

kpi_df["booking_month"] = (
    kpi_df["booking_date"]
    .dt.to_period("M")
    .astype("string")
)

monthly_booking_summary = (
    kpi_df
    .groupby("booking_month")
    .agg(
        bookings=("booking_id", "nunique"),
        total_payment=("amount", "sum")
    )
    .reset_index()
)

display(monthly_booking_summary)

,booking_month,bookings,total_payment
0,2025-04,48,404599.52
1,2025-05,80,788902.06
2,2025-06,89,561019.96
3,2025-07,66,588025.39
4,2025-08,101,871354.45
5,2025-09,75,559557.99
6,2025-10,76,601803.08
7,2025-11,90,509170.91
8,2025-12,89,718387.40
9,2026-01,89,721234.44


In [119]:
# 11.14: Export KPI Tables

kpi_summary.to_csv(
    output_dir / "kpi_summary.csv",
    index=False
)

airline_summary.to_csv(
    output_dir / "airline_summary.csv",
    index=False
)

route_summary.to_csv(
    output_dir / "route_summary.csv",
    index=False
)

duration_summary_by_airline.to_csv(
    output_dir / "duration_by_airline.csv",
    index=False
)

overnight_summary.to_csv(
    output_dir / "overnight_summary.csv",
    index=False
)

anomaly_summary.to_csv(
    output_dir / "anomaly_summary.csv",
    index=False
)

booking_status_summary.to_csv(
    output_dir / "booking_status_summary.csv",
    index=False
)

age_summary.to_csv(
    output_dir / "age_summary.csv",
    index=False
)

monthly_booking_summary.to_csv(
    output_dir / "monthly_booking_summary.csv",
    index=False
)

print("All KPI tables exported successfully.")

All KPI tables exported successfully.


In [121]:
# 11.15: Final Analytical Dataset Exporting

analytical_bi = kpi_df.copy()

analytical_bi.to_csv(
    output_dir / "airline_analytical_data.csv",
    index=False
)

print("Final analytical dataset updated.")
print("Shape:", analytical_bi.shape)

Final analytical dataset updated.
Shape: (1420, 29)


In [122]:
# 11.16: Listing Output Files

for file in sorted(output_dir.iterdir()):
    print(file.name)

age_summary.csv
airline_analytical_data.csv
airline_summary.csv
anomaly_summary.csv
booking_status_summary.csv
bookings_protected.csv
duration_by_airline.csv
flights_clean.csv
kpi_summary.csv
monthly_booking_summary.csv
overnight_summary.csv
passengers_protected.csv
payments_clean.csv
route_summary.csv


12. Export & Final Pipeline Packaging

In [123]:
# 12.1: Final dataset cleanup

# Create final export copy
final_bi = analytical_bi.copy()

# Remove temporary/helper columns if they exist
temporary_columns = [
    "flight_prefix",
    "airline_from_id",
    "airline_expected",
    "airline_match",
    "flight_id_valid",
    "departure_valid",
    "arrival_valid",
    "passenger_exists",
    "flight_exists",
    "booking_exists",
    "amount_valid",
    "status_valid"
]

final_bi = final_bi.drop(
    columns=[c for c in temporary_columns if c in final_bi.columns],
    errors="ignore"
)

print("Final BI dataset shape:", final_bi.shape)
display(final_bi.head())

Final BI dataset shape: (1420, 29)


,booking_id,passenger_id_hash,flight_id,booking_date,status,seat_number,airline,source,destination,departure_time,...,first_name,last_name,age,gender,aadhaar_masked,phone_masked,email_masked,flight_day_type,age_group,booking_month
0,B1000,977c3154ed97450c76e731cfa64c1b4071ddd7001d11e1...,AI192,2025-06-14 11:37:36.951,CANCELLED,3D,Air India,MAA,BOM,2026-04-20 23:05:41.703,...,Krishna,Mehta,4,M,55*********,+9************,kr**************************,Unknown,Under 18,2025-06
1,B1001,604c9afd067fac61e2daa8a506a2030f2d8c48a0c123a8...,6F026,2025-11-02 11:37:36.951,CANCELLED,18A,IndiGo,BOM,CCU,2026-04-20 22:46:42.000,...,Krishna,Singh,82,F,28*********,+9************,kr***********************,Unknown,60+,2025-11
2,B1002,906ea2a505d51fd41a2b2f02d9dfc962d3831101cd8232...,SJ010,2025-08-25 11:37:36.951,CANCELLED,30C,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,...,Arjun,Banerjee,62,F,44**********,+9************,ar************************,Unknown,60+,2025-08
3,B1002,906ea2a505d51fd41a2b2f02d9dfc962d3831101cd8232...,SJ010,2025-08-25 11:37:36.951,CANCELLED,30C,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,...,Arjun,Banerjee,62,F,44**********,+9************,ar************************,Unknown,60+,2025-08
4,B1003,253e3decf26bba87ebad66473c9ad56664e004ace4b3cf...,AI069,2025-12-30 11:37:36.951,CONFIRMED,33A,Air India,DEL,BOM,2026-04-20 22:43:41.702,...,Aarav,Ghosh,60,M,95**********,+9************,aa**********************,Unknown,46-60,2025-12


In [124]:
# 12.2: Final data-quality check

print("Final dataset validation!!\n")

print("Rows:", len(final_bi))
print("Columns:", len(final_bi.columns))
print("Duplicate booking IDs:", final_bi["booking_id"].duplicated().sum())
print("Missing booking IDs:", final_bi["booking_id"].isna().sum())
print("Missing flight IDs:", final_bi["flight_id"].isna().sum())
print("Missing airline:", final_bi["airline"].isna().sum())
print("Missing source:", final_bi["source"].isna().sum())
print("Missing destination:", final_bi["destination"].isna().sum())

Final dataset validation!!

Rows: 1420
Columns: 29
Duplicate booking IDs: 420
Missing booking IDs: 0
Missing flight IDs: 0
Missing airline: 0
Missing source: 0
Missing destination: 0


In [125]:
# 12.3: Export the main BI dataset

final_bi_path = output_dir / "airlines_analytical_bi.csv"

final_bi.to_csv(
    final_bi_path,
    index=False
)

print(f"Saved: {final_bi_path}")

Saved: /content/output/airlines_analytical_bi.csv


In [126]:
# 12.4: Exporting individual cleaned datasets

flights_final.to_csv(
    output_dir / "flights_cleaned.csv",
    index=False
)

bookings_final.to_csv(
    output_dir / "bookings_cleaned.csv",
    index=False
)

payments_final.to_csv(
    output_dir / "payments_cleaned.csv",
    index=False
)

passengers_final.to_csv(
    output_dir / "passengers_protected.csv",
    index=False
)

print("Cleaned datasets exported successfully.")

Cleaned datasets exported successfully.


In [129]:
# 12.5: Exporting KPI Tables

kpi_summary.to_csv(
    output_dir / "kpi_summary.csv",
    index=False
)

airline_summary.to_csv(
    output_dir / "airline_summary.csv",
    index=False
)

route_summary.to_csv(
    output_dir / "route_summary.csv",
    index=False
)

duration_summary.to_csv(
    output_dir / "duration_summary.csv",
    index=False
)

overnight_summary.to_csv(
    output_dir / "overnight_summary.csv",
    index=False
)

anomaly_summary.to_csv(
    output_dir / "anomaly_summary.csv",
    index=False
)

airline_anomaly.to_csv(
    output_dir / "airline_anomaly_summary.csv",
    index=False
)

booking_status_summary.to_csv(
    output_dir / "booking_status_summary.csv",
    index=False
)

monthly_booking_summary.to_csv(
    output_dir / "monthly_booking_summary.csv",
    index=False
)

print("KPI tables exported successfully.")

KPI tables exported successfully.


In [130]:
# 12.6: Checking main Power BI file

check_df = pd.read_csv(
    output_dir / "airlines_analytical_bi.csv"
)

print("Power BI dataset:")
print(check_df.shape)

display(check_df.head())

Power BI dataset:
(1420, 29)


,booking_id,passenger_id_hash,flight_id,booking_date,status,seat_number,airline,source,destination,departure_time,...,first_name,last_name,age,gender,aadhaar_masked,phone_masked,email_masked,flight_day_type,age_group,booking_month
0,B1000,977c3154ed97450c76e731cfa64c1b4071ddd7001d11e1...,AI192,2025-06-14 11:37:36.951,CANCELLED,3D,Air India,MAA,BOM,2026-04-20 23:05:41.703,...,Krishna,Mehta,4,M,55*********,+9************,kr**************************,Unknown,Under 18,2025-06
1,B1001,604c9afd067fac61e2daa8a506a2030f2d8c48a0c123a8...,6F026,2025-11-02 11:37:36.951,CANCELLED,18A,IndiGo,BOM,CCU,2026-04-20 22:46:42.000,...,Krishna,Singh,82,F,28*********,+9************,kr***********************,Unknown,60+,2025-11
2,B1002,906ea2a505d51fd41a2b2f02d9dfc962d3831101cd8232...,SJ010,2025-08-25 11:37:36.951,CANCELLED,30C,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,...,Arjun,Banerjee,62,F,44**********,+9************,ar************************,Unknown,60+,2025-08
3,B1002,906ea2a505d51fd41a2b2f02d9dfc962d3831101cd8232...,SJ010,2025-08-25 11:37:36.951,CANCELLED,30C,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,...,Arjun,Banerjee,62,F,44**********,+9************,ar************************,Unknown,60+,2025-08
4,B1003,253e3decf26bba87ebad66473c9ad56664e004ace4b3cf...,AI069,2025-12-30 11:37:36.951,CONFIRMED,33A,Air India,DEL,BOM,2026-04-20 22:43:41.702,...,Aarav,Ghosh,60,M,95**********,+9************,aa**********************,Unknown,46-60,2025-12


13. Pipeline Summary